In [0]:
-- 1. Parse JSON, Type, Trim (Enabled for Streaming)
CREATE TEMPORARY STREAMING TABLE stg_inventory_typed AS
SELECT
  TRIM(payload.event_id) AS event_id,
  
  COALESCE(
    try_to_date(TRIM(payload.restock_date), 'M/d/yyyy'),
    try_to_date(TRIM(payload.restock_date), 'yyyy-M-d'),
    try_to_date(TRIM(payload.restock_date), 'd-MMM-yy'),
    CAST(TRIM(payload.restock_date) AS DATE)
  ) AS restock_date,
  
  CAST(TRIM(payload.product_id) AS INT) AS product_id,
  CAST(TRIM(payload.store_id) AS INT) AS store_id,
  CAST(TRIM(payload.restock_quantity) AS INT) AS restock_quantity,
  
  event_timestamp AS _ingestion_timestamp
FROM (
  SELECT 
    from_json(
      message_payload, 
      'event_id STRING, restock_date STRING, product_id STRING, store_id STRING, restock_quantity STRING'
    ) AS payload,
    event_timestamp
  FROM STREAM(mavenmarket.bronze.kafka_inventory)
);

-- 2. Run KPI Checks & ACTIVE SK LOOKUP
CREATE TEMPORARY STREAMING TABLE stg_inventory_checked AS
SELECT 
  i.event_id,
  i.restock_date,
  i.restock_quantity,
  i._ingestion_timestamp,
  
  -- Pull in the Surrogate Keys from the active dimensions
  p.product_sk,
  s.store_sk,
  
  -- HARD CHECKS
  CASE WHEN i.event_id IS NULL THEN 'Completeness: event_id is null; ' ELSE '' END AS err_event_id,
  CASE WHEN i.restock_quantity IS NULL THEN 'Completeness: restock_quantity is null; ' ELSE '' END AS err_qty_null,
  CASE WHEN i.restock_quantity <= 0 THEN 'Validity: restock_quantity must be greater than 0; ' ELSE '' END AS err_qty_neg,
  
  -- Referential Integrity: Checked against the returned SKs
  CASE WHEN i.product_id IS NOT NULL AND p.product_sk IS NULL 
       THEN 'Referential Integrity: product not active; ' ELSE '' END AS err_fk_product,
  CASE WHEN i.store_id IS NOT NULL AND s.store_sk IS NULL 
       THEN 'Referential Integrity: store not active; ' ELSE '' END AS err_fk_store,
       
  -- WARN CHECKS
  CASE WHEN i.restock_date IS NULL THEN 'Completeness: restock_date is null; ' ELSE '' END AS warn_date
FROM STREAM(LIVE.stg_inventory_typed) i

-- Join to active Silver Products (Current version only)
LEFT JOIN LIVE.silver_products p 
  ON i.product_id = p.product_id 
  AND p.__END_AT IS NULL

-- Join to active Silver Stores (Current version only)
LEFT JOIN LIVE.silver_stores s 
  ON i.store_id = s.store_id 
  AND s.__END_AT IS NULL;

-- 3. Consolidate Flags
CREATE TEMPORARY STREAMING TABLE stg_inventory_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_event_id, err_qty_null, err_qty_neg, err_fk_product, err_fk_store)) AS hard_failures,
  TRIM(CONCAT(warn_date)) AS dq_flags
FROM STREAM(LIVE.stg_inventory_checked);

-- 4. Target Silver Table (Valid Records)
CREATE STREAMING TABLE silver_inventory
COMMENT "Cleaned Kafka inventory restocks with Surrogate Keys and warn-level data quality flags."
AS SELECT 
  * EXCEPT(err_event_id, err_qty_null, err_qty_neg, err_fk_product, err_fk_store, warn_date, hard_failures)
FROM STREAM(LIVE.stg_inventory_flagged)
WHERE hard_failures = '';

-- 5. Target Quarantine Table (Invalid Records)
CREATE STREAMING TABLE quarantine_inventory
COMMENT "Quarantined inventory failing hard checks"
AS SELECT 
  * EXCEPT(err_event_id, err_qty_null, err_qty_neg, err_fk_product, err_fk_store, warn_date, dq_flags),
  hard_failures AS quarantine_reasons
FROM STREAM(LIVE.stg_inventory_flagged)
WHERE hard_failures != '';